# Act II — The Great Divergence: Who Benefited and Who Didn't
### *The Diverging American Dream: Can America Afford a Family?*

**Story beat:** After 1979, the American economy kept growing — but the gains
stopped being shared. The bottom 90% of workers saw just **+43.7%** real wage growth
over 44 years. The top 1% saw **+181.7%**. The top 0.1% saw **+353.9%**.
Meanwhile income inequality, measured by the Gini coefficient, rose almost every
single year. This act answers: *who was inside the prosperity, and who was watching
from the outside?*

This notebook builds **5 visualizations** across two tonal registers:
the first two are on a **dark background** (signaling the data gets grimmer);
the last three return to the light background for context and comparison.

| # | Visualization | Type | Data |
|---|---|---|---|
| 1 | Wage Growth by Income Group Since 1979 | Interactive dark multi-line | EPI annual wages |
| 2 | The 90/10 Wage Gap Widens | Interactive dark area chart | EPI hourly percentiles |
| 3 | Gini Coefficient — Inequality Rising | Static annotated line | GINIALLRH |
| 4 | Top 1% Total Assets vs. Real Median Income | Static dual-axis | WFRBLT01000 + MEFAINUSA672N |
| 5 | Wage Growth Divergence — Bar Race Summary | Static grouped bar | EPI annual wages |

---


## Setup — Imports, Theme, Data Loading

In [1]:
import pandas as pd
import numpy as np
import plotly.graph_objects as go
import plotly.io as pio
from plotly.subplots import make_subplots
import warnings
warnings.filterwarnings('ignore')

from theme import (
    THEME, C,
    apply_theme, apply_dark_theme,
    annotate_threshold, add_recession_bands, watermark,
    apply_mpl_style, PLOTLY_LAYOUT, PLOTLY_LAYOUT_DARK
)

RAW = "/Users/ishika/Desktop/MS/Scholarship_Project/Data/"   # adjust to match your folder structure

print("✓ Imports complete")
print(f"  Dark bg  : {C.bg_dark}  ← Acts II uses this")
print(f"  Red      : {C.red}")
print(f"  Blue     : {C.blue}")
print(f"  Gold     : {C.gold}")


✓ Imports complete
  Dark bg  : #0F0E0B  ← Acts II uses this
  Red      : #B84A2E
  Blue     : #2A6FAF
  Gold     : #C8872A


## Load & Prepare All Act II Data

In [2]:
# ── EPI date parser (handles 2-digit years correctly) ────────
def parse_epi_year(date_str):
    """
    EPI dates look like '1/1/73' or '1/1/24'.
    Rule: yr <= 30 → 2000s, yr > 30 → 1900s.
    """
    yr = int(str(date_str).split('/')[-1])
    if yr < 100:
        return (2000 + yr) if yr <= 30 else (1900 + yr)
    return yr


# ── FRED loader (same as Act 1) ───────────────────────────────
def load_fred(filename, col_rename=None, start_year=1960, end_year=2025):
    df = pd.read_csv(RAW + filename)
    # Parse M/D/YY explicitly and correct rollover years (e.g., 67 -> 2067).
    df['DATE'] = pd.to_datetime(df['DATE'], format='%m/%d/%y', errors='coerce')
    rollover = df['DATE'].dt.year > end_year
    if rollover.any():
        df.loc[rollover, 'DATE'] = df.loc[rollover, 'DATE'] - pd.DateOffset(years=100)
    df['year'] = df['DATE'].dt.year
    val_col = [c for c in df.columns if c not in ['DATE', 'year']][0]
    df = df.rename(columns={val_col: col_rename or val_col})
    name = col_rename or val_col
    df = df.dropna(subset=[name])
    df = df[(df['year'] >= start_year) & (df['year'] <= end_year)]
    return df.groupby('year')[name].mean()


# ─────────────────────────────────────────────────────────────
# EPI Hourly Wage Percentiles (real 2023 $, 1973–2025)
# ─────────────────────────────────────────────────────────────
wp_raw = pd.read_csv(RAW + 'epi_wage_percentiles.csv')
wp_raw['year'] = wp_raw['date'].apply(parse_epi_year)
wp_raw = wp_raw.set_index('year').drop(columns='date').sort_index()
wp_raw.columns = ['p10', 'p20', 'p30', 'p40', 'p50', 'p60', 'p70', 'p80', 'p90']

# ─────────────────────────────────────────────────────────────
# EPI Annual Wages by Group (real 2023 $, 1951–2023)
# ─────────────────────────────────────────────────────────────
aw_raw = pd.read_csv(RAW + 'epi_annual_wages.csv')
aw_raw['year'] = aw_raw['date'].apply(parse_epi_year)
aw_raw = aw_raw.set_index('year').drop(columns='date').sort_index()
aw_raw.columns = [
    'bottom_90', 'top_10', 'top_5', 'top_1', 'top_0_1',
    'p90_95', 'p95_99', 'p90_99', 'p99_999'
]

# ─────────────────────────────────────────────────────────────
# FRED Series
# ─────────────────────────────────────────────────────────────
gini    = load_fred('GINIALLRH.csv',    'gini',             start_year=1967)
wfr     = load_fred('WFRBLT01000.csv',  'top1_assets_m',    start_year=1990)
income  = load_fred('MEFAINUSA672N.csv','real_median_fam_income', start_year=1960)
hhi     = load_fred('MEHOINUSA672N.csv','real_median_hh_income',  start_year=1984)
weekly  = load_fred('LES1252881600Q.csv','median_weekly_earnings', start_year=1979)
comp    = load_fred('COMPRNFB.csv',     'real_compensation', start_year=1960)

# Top 1% assets in trillions
top1_trillions = wfr / 1e6

# ─────────────────────────────────────────────────────────────
# Index wages to 1979 = 100 (the structural break year)
# ─────────────────────────────────────────────────────────────
BASE = 1979

def idx100(series, base_year=BASE):
    base = series.loc[base_year]
    return (series / base * 100).round(2)

wp_idx = wp_raw.apply(lambda col: idx100(col.dropna()))

aw_idx = aw_raw.apply(lambda col: idx100(col.dropna(), base_year=BASE)
                      if BASE in col.dropna().index else col)

# ─────────────────────────────────────────────────────────────
# Pre-compute key headline numbers
# ─────────────────────────────────────────────────────────────
last_aw_yr  = aw_raw.index.max()
last_wp_yr  = wp_raw.index.max()
last_gini_yr = int(gini.dropna().index.max())

growth = {}
for grp in ['bottom_90', 'top_10', 'top_5', 'top_1', 'top_0_1']:
    v79   = aw_raw.loc[1979, grp]
    v_last = aw_raw.loc[last_aw_yr, grp]
    growth[grp] = round(((v_last / v79) - 1) * 100, 1)

print("✓ All Act II data loaded")
print(f"\n── Wage group growth since 1979 (to {last_aw_yr}) ──")
labels = {'bottom_90':'Bottom 90%','top_10':'Top 10%','top_5':'Top 5%',
          'top_1':'Top 1%','top_0_1':'Top 0.1%'}
for grp, pct in growth.items():
    bar = '█' * int(pct / 10)
    print(f"  {labels[grp]:12s}: +{pct:5.1f}%  {bar}")

print(f"\n── Hourly wage percentile growth since 1979 (to {last_wp_yr}) ──")
for pct_key in ['p10', 'p50', 'p90']:
    v79   = wp_raw.loc[1979, pct_key]
    v_end = wp_raw.loc[last_wp_yr, pct_key]
    g     = ((v_end / v79) - 1) * 100
    print(f"  {pct_key}: ${v79:.2f} → ${v_end:.2f}  (+{g:.1f}%)")

print(f"\n── Gini (1967→{last_gini_yr}) ──")
print(f"  1967: {gini.loc[1967]:.3f}  →  {last_gini_yr}: {gini.loc[last_gini_yr]:.3f}")

print(f"\n── Top 1% assets ──")
print(f"  1990: ${top1_trillions.loc[1990]:.1f}T  →  "
      f"{int(top1_trillions.dropna().index.max())}: "
      f"${top1_trillions.dropna().iloc[-1]:.1f}T")


✓ All Act II data loaded

── Wage group growth since 1979 (to 2023) ──
  Bottom 90%  : + 43.7%  ████
  Top 10%     : +115.0%  ███████████
  Top 5%      : +135.4%  █████████████
  Top 1%      : +181.7%  ██████████████████
  Top 0.1%    : +353.9%  ███████████████████████████████████

── Hourly wage percentile growth since 1979 (to 2025) ──
  p10: $11.32 → $14.56  (+28.6%)
  p50: $19.74 → $25.67  (+30.0%)
  p90: $39.32 → $65.38  (+66.3%)

── Gini (1967→2024) ──
  1967: 0.397  →  2024: 0.488

── Top 1% assets ──
  1990: $4.9T  →  2024: $49.5T


---
## Visualization 1 — "The Paycheck Lottery Since 1979"
**Interactive dark multi-line chart · Click legend to isolate any group**

All annual wages indexed to 100 in 1979 so every group starts at the same point.
The divergence that follows is the entire story of Act II.

> **The aha**: Starting from the same baseline, the top 0.1% line shoots to nearly
> **450** while the bottom 90% barely reaches **145**. Same economy. Different result.
> Click any legend item to isolate a single group.


In [13]:
# ── Series config for the indexed wage chart ─────────────────
WAGE_SERIES = [
    ('top_0_1',   'Top 0.1%',   C.red,        3.2),
    ('top_1',     'Top 1%',     C.red_light,  2.5),
    ('top_5',     'Top 5%',     C.orange,     2.0),
    ('top_10',    'Top 10%',    C.gold,       2.0),
    ('bottom_90', 'Bottom 90%', C.blue,       3.0),
]

fig1 = go.Figure()

# ── Recession bands (dark version) ───────────────────────────
add_recession_bands(fig1, dark=True)

# ── 1979 = 100 reference line ─────────────────────────────────
fig1.add_hline(
    y=100,
    line_dash='dot', line_color=C.gray, line_width=1.2,
    annotation_text='1979 = 100 (shared baseline)',
    annotation_position='top left',
    annotation_font=dict(size=10, color=C.gray, family=THEME['font_mono']),
)

# ── Each wage group ───────────────────────────────────────────
for key, label, color, width in WAGE_SERIES:
    s = aw_idx[key].dropna()
    last_val = float(s.iloc[-1])
    last_yr  = int(s.index.max())

    fig1.add_trace(go.Scatter(
        x=s.index.tolist(),
        y=s.values.tolist(),
        name=label,
        mode='lines',
        line=dict(color=color, width=width),
        hovertemplate=(
            f'<b>{label}</b><br>'
            'Year: %{x}<br>'
            'Index: %{y:.1f}<br>'
            f'(1979 = 100)'
            '<extra></extra>'
        ),
    ))

    # End-of-line label
    fig1.add_annotation(
        x=last_yr, y=last_val,
        text=f'<b>{label}</b><br>{last_val:.0f}',
        showarrow=False,
        xanchor='left', xshift=8,
        font=dict(size=10, color=color, family=THEME['font_mono']),
    )

# ── Annotate the divergence visually ─────────────────────────
# Arrow from Bottom 90 endpoint to Top 0.1 endpoint
yr_arrow = last_aw_yr
bot_val = float(aw_idx.loc[yr_arrow, 'bottom_90'])
top_val = float(aw_idx.loc[yr_arrow, 'top_0_1'])
fig1.add_annotation(
    x=yr_arrow - 5, y=(bot_val + top_val) / 2,
    text=f'<b>Gap: {top_val - bot_val:.0f} pts</b><br>same start, 1979',
    showarrow=False,
    font=dict(size=11, color=THEME['text_light'], family=THEME['font']),
    bgcolor=THEME['surface_dark'],
    bordercolor=C.gold, borderwidth=1,
    align='center',
)

apply_dark_theme(fig1)

fig1.update_layout(
    title=dict(
        text='<b>The Paycheck Lottery Since 1979</b><br>'
             '<sup>Real annual wages indexed to 100 in 1979 · '
             'All groups start at the same point · Click legend to isolate</sup>',
        font=dict(size=THEME['font_size']['lg'], family=THEME['font'],
                  color=THEME['text_light']),
    ),
        # xaxis=dict(
        #     title='Year',
        #     range=[1979, last_aw_yr + 3],
        #     tickmode='linear', dtick=5,
        #     color=THEME['text_muted_dark'],
        # ),
    yaxis=dict(
        title='Wage Index',
        range=[80, top_val * 1.5],
        color=THEME['text_muted_dark'],
    ),
    legend=dict(
        orientation='v',
        x=0.01, y=0.99,
        xanchor='left', yanchor='top',
        bgcolor='rgba(0,0,0,0)',
        font=dict(color=THEME['text_light'], size=11),
    ),
    height=520,
    margin=dict(t=100, b=70, l=70, r=180),
)


fig1.show()

# Print key growth stats
print("\n── Growth from 1979 baseline ──────────────────────────────────────")
for key, label, *_ in WAGE_SERIES:
    idx_val = float(aw_idx.loc[last_aw_yr, key]) if last_aw_yr in aw_idx.index else None
    if idx_val:
        print(f"  {label:12s}: index = {idx_val:.1f}  (+{idx_val-100:.1f}% since 1979)")



── Growth from 1979 baseline ──────────────────────────────────────
  Top 0.1%    : index = 453.9  (+353.9% since 1979)
  Top 1%      : index = 281.7  (+181.7% since 1979)
  Top 5%      : index = 235.4  (+135.4% since 1979)
  Top 10%     : index = 215.0  (+115.0% since 1979)
  Bottom 90%  : index = 143.7  (+43.7% since 1979)


---
## Visualization 2 — "The Gap Between the Top and the Bottom"
**Interactive dark area chart**

This chart shows the **absolute dollar spread** between the 90th and 10th
percentile hourly worker — real dollars, the same purchasing power in every year.

The shaded gap between the two lines *is* the story of wage inequality.
Hover to see the exact values and gap in any year.

> **The mechanic**: The bottom line barely moves. The top line climbs steadily.
> The gap that opens between them is time stolen from the bottom by the top —
> not through malice, but through structural changes in how the economy distributes
> its gains.


In [14]:
fig2 = go.Figure()

years_wp = wp_raw.index.tolist()

p10_vals = wp_raw['p10'].tolist()
p50_vals = wp_raw['p50'].tolist()
p90_vals = wp_raw['p90'].tolist()

# ── Filled gap between p90 and p10 ───────────────────────────
fig2.add_trace(go.Scatter(
    x=years_wp + years_wp[::-1],
    y=p90_vals + p10_vals[::-1],
    fill='toself',
    fillcolor='rgba(184,74,46,0.15)',
    line=dict(width=0),
    showlegend=False,
    hoverinfo='skip',
    name='Gap area',
))

# ── P10 line ─────────────────────────────────────────────────
fig2.add_trace(go.Scatter(
    x=years_wp, y=p10_vals,
    name='10th Percentile',
    mode='lines',
    line=dict(color=C.blue, width=2.8),
    hovertemplate='<b>10th %ile</b> · %{x}<br>$%{y:.2f}/hr<extra></extra>',
))

# ── P50 line ─────────────────────────────────────────────────
fig2.add_trace(go.Scatter(
    x=years_wp, y=p50_vals,
    name='50th Percentile (Median)',
    mode='lines',
    line=dict(color=C.gold, width=2.5, dash='dot'),
    hovertemplate='<b>Median</b> · %{x}<br>$%{y:.2f}/hr<extra></extra>',
))

# ── P90 line ─────────────────────────────────────────────────
fig2.add_trace(go.Scatter(
    x=years_wp, y=p90_vals,
    name='90th Percentile',
    mode='lines',
    line=dict(color=C.red, width=2.8),
    hovertemplate='<b>90th %ile</b> · %{x}<br>$%{y:.2f}/hr<extra></extra>',
))

# ── Invisible trace for gap tooltip ──────────────────────────
gap_vals = [p90 - p10 for p90, p10 in zip(p90_vals, p10_vals)]
fig2.add_trace(go.Scatter(
    x=years_wp,
    y=[(p90 + p10) / 2 for p90, p10 in zip(p90_vals, p10_vals)],
    mode='markers',
    marker=dict(opacity=0, size=8),
    name='90–10 Gap',
    customdata=gap_vals,
    hovertemplate='<b>90–10 Gap</b> · %{x}<br>$%{customdata:.2f}/hr<extra></extra>',
))

# ── Annotations: 1979 and latest gap ─────────────────────────
gap_1979 = wp_raw.loc[1979, 'p90'] - wp_raw.loc[1979, 'p10']
gap_last = wp_raw.loc[last_wp_yr, 'p90'] - wp_raw.loc[last_wp_yr, 'p10']
mid_1979 = (wp_raw.loc[1979, 'p90'] + wp_raw.loc[1979, 'p10']) / 2
mid_last = (wp_raw.loc[last_wp_yr, 'p90'] + wp_raw.loc[last_wp_yr, 'p10']) / 2

fig2.add_annotation(
    x=1979, y=mid_1979,
    text=f'Gap: ${gap_1979:.2f}/hr',
    showarrow=True, arrowhead=2, arrowcolor=C.gold,
    font=dict(size=10, color=C.gold, family=THEME['font_mono']),
    bgcolor=THEME['surface_dark'],
    ax=50, ay=0,
)
fig2.add_annotation(
    x=last_wp_yr, y=mid_last,
    text=f'Gap: ${gap_last:.2f}/hr',
    showarrow=True, arrowhead=2, arrowcolor=C.red,
    font=dict(size=10, color=C.red, family=THEME['font_mono']),
    bgcolor=THEME['surface_dark'],
    ax=-50, ay=0,
)

add_recession_bands(fig2, dark=True)
apply_dark_theme(fig2)

fig2.update_layout(
    title=dict(
        text='<b>The Gap Between the Top and the Bottom</b><br>'
             '<sup>Real hourly wages by percentile, 1973–2025 · '
             'Shaded area = the inequality gap · Hover for exact values</sup>',
        font=dict(size=THEME['font_size']['lg'], family=THEME['font'],
                  color=THEME['text_light']),
    ),
    xaxis=dict(
        title='Year',
        range=[1973, last_wp_yr + 1],
        tickmode='linear', dtick=5,
        color=THEME['text_muted_dark'],
    ),
    yaxis=dict(
        title='Real Hourly Wage (2023 $)',
        tickformat='$,.0f',
        color=THEME['text_muted_dark'],
    ),
    legend=dict(
        orientation='h', x=0.01, y=1.08,
        font=dict(color=THEME['text_light'], size=11),
        bgcolor='rgba(0,0,0,0)',
    ),
    height=500,
    margin=dict(t=110, b=70, l=80, r=40),
)

watermark(fig2,
          f'Source: Economic Policy Institute · epi_wage_percentiles.csv · '
          f'Real 2023 dollars',
          dark=True)

fig2.show()

print(f"\n── 90/10 Wage Gap over time ─────────────────────────────────────")
for yr in [1973, 1979, 1990, 2000, 2010, 2020, last_wp_yr]:
    if yr in wp_raw.index:
        gap = wp_raw.loc[yr,'p90'] - wp_raw.loc[yr,'p10']
        ratio = wp_raw.loc[yr,'p90'] / wp_raw.loc[yr,'p10']
        print(f"  {yr}: ${gap:.2f}/hr gap  ({ratio:.2f}× ratio)")



── 90/10 Wage Gap over time ─────────────────────────────────────
  1973: $28.99/hr gap  (3.78× ratio)
  1979: $27.99/hr gap  (3.47× ratio)
  1990: $32.97/hr gap  (4.44× ratio)
  2000: $36.53/hr gap  (4.40× ratio)
  2010: $42.30/hr gap  (4.70× ratio)
  2020: $51.31/hr gap  (4.80× ratio)
  2025: $50.82/hr gap  (4.49× ratio)


---
## Visualization 3 — "The Inequality Ratchet"
**Static annotated line chart · Light background**

The Gini coefficient is the most widely used single measure of income inequality.
Zero means perfect equality; one means total concentration.
The U.S. Gini has risen in almost every decade since 1967.

> **The ratchet**: Inequality rose during recessions and rarely came back down.
> Each crisis — 1973 oil shock, 1980s Volcker recessions, 2001 dot-com bust,
> 2008 Great Recession — left the distribution a little more unequal than before.


In [15]:
fig3 = go.Figure()

gini_clean = gini.dropna()
years_g = gini_clean.index.tolist()
vals_g  = gini_clean.values.tolist()

# ── Area fill ─────────────────────────────────────────────────
fig3.add_trace(go.Scatter(
    x=years_g + years_g[::-1],
    y=vals_g + [min(vals_g)] * len(years_g),
    fill='toself',
    fillcolor=C.red_fill,
    line=dict(width=0),
    showlegend=False, hoverinfo='skip',
))

# ── Gini line ─────────────────────────────────────────────────
fig3.add_trace(go.Scatter(
    x=years_g,
    y=vals_g,
    mode='lines',
    name='Gini Coefficient',
    line=dict(color=C.red, width=3),
    hovertemplate='<b>%{x}</b><br>Gini: %{y:.3f}<extra></extra>',
))

# ── Trend line (OLS) ─────────────────────────────────────────
x_arr = np.array(years_g)
y_arr = np.array(vals_g)
m, b  = np.polyfit(x_arr, y_arr, 1)
trend = m * x_arr + b
fig3.add_trace(go.Scatter(
    x=years_g, y=trend.tolist(),
    mode='lines',
    name=f'Trend (+{m*10:.4f}/decade)',
    line=dict(color=C.gold, width=1.5, dash='dot'),
    hoverinfo='skip',
))

# ── Key event annotations ─────────────────────────────────────
events = [
    (1967, 0.397, '1967: 0.397<br>Start of record', -20, -45),
    (1993, 0.454, '1993: 0.454', 20, -40),
    (2011, 0.477, '2011: post-GR<br>peak 0.477', 20, -45),
    (last_gini_yr, float(gini.loc[last_gini_yr]),
     f'{last_gini_yr}: {float(gini.loc[last_gini_yr]):.3f}', -20, -45),
]
for yr, val, text, ax, ay in events:
    if yr in gini_clean.index:
        fig3.add_annotation(
            x=yr, y=val,
            text=text,
            showarrow=True, arrowhead=2, arrowcolor=C.red,
            arrowwidth=1.5,
            font=dict(size=10, color=C.text, family=THEME['font_mono']),
            bgcolor=THEME['background'],
            bordercolor=C.red, borderwidth=1,
            ax=ax, ay=ay,
        )

# ── Annotate total change ─────────────────────────────────────
total_rise = float(gini_clean.iloc[-1]) - float(gini_clean.iloc[0])
fig3.add_annotation(
    x=1990, y=0.415,
    text=f'<b>+{total_rise:.3f}</b> total rise<br>since 1967',
    showarrow=False,
    font=dict(size=13, color=C.red, family=THEME['font']),
    bgcolor=THEME['background'],
    bordercolor=C.red, borderwidth=1,
)

add_recession_bands(fig3)
apply_theme(fig3)

fig3.update_layout(
    title=dict(
        text='<b>The Inequality Ratchet</b><br>'
             '<sup>U.S. Gini Coefficient, 1967–2024 · '
             '0 = perfect equality · 1 = maximum inequality</sup>',
        font=dict(size=THEME['font_size']['lg'], family=THEME['font']),
    ),
    xaxis=dict(title='Year', range=[1967, last_gini_yr + 1],
               tickmode='linear', dtick=5),
    yaxis=dict(
        title='Gini Coefficient',
        range=[0.37, 0.52],
        tickformat='.3f',
    ),
    showlegend=True,
    legend=dict(x=0.01, y=0.99, xanchor='left', yanchor='top'),
    height=460,
    margin=dict(t=100, b=70, l=70, r=40),
)

watermark(fig3, 'Source: U.S. Census Bureau via FRED · GINIALLRH · '
                'All races, all households')
fig3.show()

gini_1967 = float(gini.loc[1967])
gini_last = float(gini.loc[last_gini_yr])
print(f"\n── Gini summary ──────────────────────────────────────────────────")
print(f"  1967 : {gini_1967:.3f}")
print(f"  {last_gini_yr}: {gini_last:.3f}")
print(f"  Rise : +{gini_last - gini_1967:.3f}  over {last_gini_yr-1967} years")
print(f"  Trend: +{m*10:.4f} per decade  (OLS fit)")
print(f"  Years declining: {sum(1 for i in range(1, len(vals_g)) if vals_g[i] < vals_g[i-1])}"
      f" of {len(vals_g)-1}")



── Gini summary ──────────────────────────────────────────────────
  1967 : 0.397
  2024: 0.488
  Rise : +0.091  over 57 years
  Trend: +0.0198 per decade  (OLS fit)
  Years declining: 15 of 57


---
## Visualization 4 — "Two Different Economies"
**Static dual-axis chart**

The most visceral single chart in Act II.
**Left axis**: Real median family income — the typical American household (2023 $).
**Right axis**: Total assets held by the top 1% — in *trillions* of dollars.

Both lines start in 1990 (the first year of Federal Reserve wealth data).
They tell completely different stories about the same economy.

> **The gut punch**: Between 1990 and 2024, real median family income rose from
> ~$63,000 to ~$105,000 — a meaningful gain. But top 1% total assets rose from
> $4.9 trillion to nearly $50 trillion. The scale of that accumulation is not just
> quantitatively different — it represents a fundamentally separate economic reality.


In [16]:
fig4 = make_subplots(specs=[[{'secondary_y': True}]])

# Align years where both series exist
both_years = sorted(set(top1_trillions.dropna().index) & set(income.index))

t1_vals  = [float(top1_trillions.loc[yr]) for yr in both_years]
inc_vals = [float(income.loc[yr]) for yr in both_years]

# ── Median income fill ────────────────────────────────────────
fig4.add_trace(go.Scatter(
    x=both_years, y=inc_vals,
    fill='tozeroy',
    fillcolor=C.blue_fill,
    line=dict(width=0),
    showlegend=False, hoverinfo='skip',
), secondary_y=False)

# ── Median income line ────────────────────────────────────────
fig4.add_trace(go.Scatter(
    x=both_years, y=inc_vals,
    name='Real Median Family Income (left)',
    mode='lines',
    line=dict(color=C.blue, width=3),
    hovertemplate='<b>Median Income</b> · %{x}<br>$%{y:,.0f}<extra></extra>',
), secondary_y=False)

# ── Top 1% assets line ────────────────────────────────────────
fig4.add_trace(go.Scatter(
    x=both_years, y=t1_vals,
    name='Top 1% Total Assets, $T (right)',
    mode='lines',
    line=dict(color=C.red, width=3),
    hovertemplate='<b>Top 1% Assets</b> · %{x}<br>$%{y:.1f}T<extra></extra>',
), secondary_y=True)

# ── Recession bands ───────────────────────────────────────────
add_recession_bands(fig4)

# ── Callout annotations ───────────────────────────────────────
yr_start = both_years[0]
yr_end   = both_years[-1]

fig4.add_annotation(
    x=yr_start,
    y=float(top1_trillions.loc[yr_start]),
    yref='y2',
    text=f'1990:<br>${float(top1_trillions.loc[yr_start]):.1f}T',
    showarrow=True, arrowhead=2, arrowcolor=C.red,
    font=dict(size=10, color=C.red, family=THEME['font_mono']),
    bgcolor=THEME['background'], bordercolor=C.red, borderwidth=1,
    ax=30, ay=-35,
)
fig4.add_annotation(
    x=yr_end,
    y=float(top1_trillions.loc[yr_end]),
    yref='y2',
    text=f'{yr_end}:<br>${float(top1_trillions.loc[yr_end]):.1f}T',
    showarrow=True, arrowhead=2, arrowcolor=C.red,
    font=dict(size=10, color=C.red, family=THEME['font_mono']),
    bgcolor=THEME['background'], bordercolor=C.red, borderwidth=1,
    ax=-40, ay=-35,
)
fig4.add_annotation(
    x=yr_start,
    y=float(income.loc[yr_start]),
    yref='y',
    text=f'1990:<br>${float(income.loc[yr_start]):,.0f}',
    showarrow=True, arrowhead=2, arrowcolor=C.blue,
    font=dict(size=10, color=C.blue, family=THEME['font_mono']),
    bgcolor=THEME['background'], bordercolor=C.blue, borderwidth=1,
    ax=30, ay=35,
)
fig4.add_annotation(
    x=yr_end,
    y=float(income.loc[yr_end]),
    yref='y',
    text=f'{yr_end}:<br>${float(income.loc[yr_end]):,.0f}',
    showarrow=True, arrowhead=2, arrowcolor=C.blue,
    font=dict(size=10, color=C.blue, family=THEME['font_mono']),
    bgcolor=THEME['background'], bordercolor=C.blue, borderwidth=1,
    ax=-50, ay=35,
)

apply_theme(fig4)

fig4.update_layout(
    title=dict(
        text='<b>Two Different Economies</b><br>'
             '<sup>Blue (left): Real Median Family Income · '
             'Red (right): Total assets held by top 1% of households</sup>',
        font=dict(size=THEME['font_size']['lg'], family=THEME['font']),
    ),
    xaxis=dict(title='Year', tickmode='linear', dtick=5),
    legend=dict(
        x=0.01, y=0.99, xanchor='left', yanchor='top',
    ),
    height=500,
    margin=dict(t=100, b=70, l=90, r=90),
)
fig4.update_yaxes(
    title_text='Real Median Family Income (2023 $)',
    tickformat='$,.0f',
    secondary_y=False,
)
fig4.update_yaxes(
    title_text='Top 1% Total Assets ($ Trillions)',
    tickformat='$.1f',
    ticksuffix='T',
    secondary_y=True,
)

watermark(fig4, 'Sources: FRED · MEFAINUSA672N (Census, real 2023 $) · '
                'WFRBLT01000 (Federal Reserve Distributional Financial Accounts)')
fig4.show()

inc_growth = ((income.loc[yr_end] / income.loc[yr_start]) - 1) * 100
t1_growth  = ((top1_trillions.loc[yr_end] / top1_trillions.loc[yr_start]) - 1) * 100
print(f"\n── 1990 → {yr_end} comparison ─────────────────────────────────────")
print(f"  Median family income  : +{inc_growth:.1f}%  (${income.loc[yr_start]:,.0f} → ${income.loc[yr_end]:,.0f})")
print(f"  Top 1% total assets   : +{t1_growth:.1f}%  (${top1_trillions.loc[yr_start]:.1f}T → ${top1_trillions.loc[yr_end]:.1f}T)")
print(f"  Assets grew {t1_growth/inc_growth:.1f}× faster than median income")



── 1990 → 2024 comparison ─────────────────────────────────────
  Median family income  : +36.9%  ($77,260 → $105,800)
  Top 1% total assets   : +899.6%  ($4.9T → $49.5T)
  Assets grew 24.4× faster than median income


---
## Visualization 5 — "The Growth Scorecard"
**Static grouped bar chart**

A clean static summary: cumulative real wage growth since 1979 for each group,
shown as a single bar chart. This is the most shareable, publication-ready
single-panel summary of the entire inequality story.

> **Designed to be a static visualization that earns its place at a glance.**
> No interaction needed — the bars say everything.


In [17]:
# ── Compute cumulative growth for each group ─────────────────
groups_ordered = [
    ('bottom_90', 'Bottom 90%'),
    ('top_10',    'Top 10%'),
    ('top_5',     'Top 5%'),
    ('top_1',     'Top 1%'),
    ('top_0_1',   'Top 0.1%'),
]

bar_labels  = [label for _, label in groups_ordered]
bar_growths = [growth[key] for key, _ in groups_ordered]

# Color: blue for bottom groups, gradient to red for top
bar_colors = [C.blue, C.gold, C.orange, C.red_light, C.red]

fig5 = go.Figure()

fig5.add_trace(go.Bar(
    x=bar_labels,
    y=bar_growths,
    marker=dict(
        color=bar_colors,
        line=dict(width=0),
    ),
    text=[f'+{v:.1f}%' for v in bar_growths],
    textposition='outside',
    textfont=dict(
        size=13,
        family=THEME['font_mono'],
        color=THEME['text'],
    ),
    hovertemplate='<b>%{x}</b><br>Growth since 1979: +%{y:.1f}%<extra></extra>',
    showlegend=False,
))

# ── Reference line: average across all (rough) ───────────────
avg_growth = sum(bar_growths) / len(bar_growths)
fig5.add_hline(
    y=avg_growth,
    line_dash='dot', line_color=C.gray, line_width=1.5,
    annotation_text=f'Simple avg: +{avg_growth:.0f}%',
    annotation_position='top right',
    annotation_font=dict(size=10, color=C.gray, family=THEME['font_mono']),
)

# ── Label: actual dollar amounts for bottom and top ───────────
v79_bot  = aw_raw.loc[1979, 'bottom_90']
v_end_bot = aw_raw.loc[last_aw_yr, 'bottom_90']
v79_top  = aw_raw.loc[1979, 'top_0_1']
v_end_top = aw_raw.loc[last_aw_yr, 'top_0_1']

fig5.add_annotation(
    x='Bottom 90%',
    y=bar_growths[0] + 8,
    text=f'${v79_bot:,.0f} → ${v_end_bot:,.0f}',
    showarrow=False,
    font=dict(size=9, color=C.blue, family=THEME['font_mono']),
)
fig5.add_annotation(
    x='Top 0.1%',
    y=bar_growths[-1] + 8,
    text=f'${v79_top/1e3:,.0f}K → ${v_end_top/1e3:,.0f}K',
    showarrow=False,
    font=dict(size=9, color=C.red, family=THEME['font_mono']),
)

apply_theme(fig5)

fig5.update_layout(
    title=dict(
        text=f'<b>The Growth Scorecard: Real Wage Growth Since 1979</b><br>'
             f'<sup>Cumulative % change in real annual wages, 1979–{last_aw_yr} · '
             f'All values in 2023 dollars</sup>',
        font=dict(size=THEME['font_size']['lg'], family=THEME['font']),
    ),
    xaxis=dict(title='Income Group'),
    yaxis=dict(
        title='Cumulative Real Wage Growth (%)',
        tickformat='+.0f',
        ticksuffix='%',
        range=[0, max(bar_growths) * 1.18],
    ),
    bargap=0.35,
    height=480,
    margin=dict(t=100, b=70, l=80, r=40),
)

watermark(fig5,
          f'Source: Economic Policy Institute analysis of Social Security '
          f'Administration wage data · epi_annual_wages.csv · 1979–{last_aw_yr}')
fig5.show()

print(f"\n── The Growth Scorecard (1979–{last_aw_yr}) ────────────────────────────")
for (key, label), g in zip(groups_ordered, bar_growths):
    bar = '█' * int(g / 8)
    print(f"  {label:14s}: +{g:6.1f}%  {bar}")
print(f"\n  The top 0.1% grew {bar_growths[-1]/bar_growths[0]:.1f}× faster than the bottom 90%")



── The Growth Scorecard (1979–2023) ────────────────────────────
  Bottom 90%    : +  43.7%  █████
  Top 10%       : + 115.0%  ██████████████
  Top 5%        : + 135.4%  ████████████████
  Top 1%        : + 181.7%  ██████████████████████
  Top 0.1%      : + 353.9%  ████████████████████████████████████████████

  The top 0.1% grew 8.1× faster than the bottom 90%


---
## Act II — Summary of Key Numbers

All headline statistics for Act II website callout cards.


In [20]:
print("=" * 65)
print("ACT II — KEY NUMBERS FOR WEBSITE CALLOUT CARDS")
print("=" * 65)

print(f"\n── Wage growth since 1979 (to {last_aw_yr}) ──────────────────────────")
for (key, label), g in zip(groups_ordered, bar_growths):
    v79   = aw_raw.loc[1979, key]
    v_end = aw_raw.loc[last_aw_yr, key]
    print(f"  {label:14s}: +{g:.1f}%   (${v79:>12,.0f} → ${v_end:>12,.0f})")

print(f"\n── 90/10 hourly wage ratio ──────────────────────────────────────")
for yr in [1979, 2000, 2010, 2020, last_wp_yr]:
    if yr in wp_raw.index:
        r = wp_raw.loc[yr,'p90'] / wp_raw.loc[yr,'p10']
        print(f"  {yr}: {r:.2f}×")

print(f"\n── Gini coefficient ─────────────────────────────────────────────")
print(f"  1967: {gini.loc[1967]:.3f}")
print(f"  1979: {gini.loc[1979]:.3f}")
print(f"  {last_gini_yr}: {gini.loc[last_gini_yr]:.3f}")
print(f"  Total rise: +{gini.loc[last_gini_yr]-gini.loc[1967]:.3f}")

print(f"\n── Top 1% total assets ──────────────────────────────────────────")
for yr in [1990, 2000, 2010, 2020, int(top1_trillions.dropna().index.max())]:
    if yr in top1_trillions.index and not pd.isna(top1_trillions.loc[yr]):
        print(f"  {yr}: ${top1_trillions.loc[yr]:.1f}T")

print(f"\n── Key ratio ────────────────────────────────────────────────────")
print(f"  Top 0.1% grew {bar_growths[-1]/bar_growths[0]:.1f}× faster than bottom 90%")
print(f"  Top 1% assets grew "
      f"{((top1_trillions.loc[yr_end]/top1_trillions.loc[yr_start])-1)*100:.0f}% "
      f"since 1990 vs {inc_growth:.0f}% for median income")

print()
print("=" * 65)
print("All five Act II visualizations complete.")
print("=" * 65)


ACT II — KEY NUMBERS FOR WEBSITE CALLOUT CARDS

── Wage growth since 1979 (to 2023) ──────────────────────────
  Bottom 90%    : +43.7%   ($      29,953 → $      43,035)
  Top 10%       : +115.0%   ($     116,635 → $     250,792)
  Top 5%        : +135.4%   ($     149,849 → $     352,773)
  Top 1%        : +181.7%   ($     281,932 → $     794,129)
  Top 0.1%      : +353.9%   ($     617,934 → $   2,805,105)

── 90/10 hourly wage ratio ──────────────────────────────────────
  1979: 3.47×
  2000: 4.40×
  2010: 4.70×
  2020: 4.80×
  2025: 4.49×

── Gini coefficient ─────────────────────────────────────────────
  1967: 0.397
  1979: 0.404
  2024: 0.488
  Total rise: +0.091

── Top 1% total assets ──────────────────────────────────────────
  1990: $4.9T
  2000: $11.9T
  2010: $17.8T
  2020: $34.7T
  2024: $49.5T

── Key ratio ────────────────────────────────────────────────────
  Top 0.1% grew 8.1× faster than bottom 90%
  Top 1% assets grew 900% since 1990 vs 37% for median income

All five

---
## Export Figures


In [18]:
import os

OUT_DIR = '../outputs/act2/'
os.makedirs(OUT_DIR, exist_ok=True)

figures = {
    'act2_fig1_wage_growth_indexed':  fig1,
    'act2_fig2_9010_gap':             fig2,
    'act2_fig3_gini':                 fig3,
    'act2_fig4_two_economies':        fig4,
    'act2_fig5_growth_scorecard':     fig5,
}

for name, f in figures.items():
    html_path = OUT_DIR + name + '.html'
    f.write_html(
        html_path,
        include_plotlyjs='cdn',
        full_html=False,
        config={
            'displayModeBar': True,
            'displaylogo': False,
            'modeBarButtonsToRemove': ['lasso2d', 'select2d'],
            'toImageButtonOptions': {
                'format': 'png', 'filename': name, 'scale': 2,
            },
        },
    )
    print(f"  ✓ HTML: {html_path}")

    try:
        png_path = OUT_DIR + name + '.png'
        f.write_image(png_path,
                      width=1200,
                      height=f.layout.height or 500,
                      scale=2)
        print(f"  ✓ PNG : {png_path}")
    except Exception as e:
        print(f"  ⚠ PNG skipped — {e}")

print(f"\n✅  Act II export complete → {OUT_DIR}")
print("\nFiles ready:")
for fn in sorted(os.listdir(OUT_DIR)):
    size = os.path.getsize(OUT_DIR + fn)
    print(f"   {fn:50s}  {size/1024:6.1f} KB")


  ✓ HTML: ../outputs/act2/act2_fig1_wage_growth_indexed.html
  ✓ PNG : ../outputs/act2/act2_fig1_wage_growth_indexed.png
  ✓ HTML: ../outputs/act2/act2_fig2_9010_gap.html
  ✓ PNG : ../outputs/act2/act2_fig2_9010_gap.png
  ✓ HTML: ../outputs/act2/act2_fig3_gini.html
  ✓ PNG : ../outputs/act2/act2_fig3_gini.png
  ✓ HTML: ../outputs/act2/act2_fig4_two_economies.html
  ✓ PNG : ../outputs/act2/act2_fig4_two_economies.png
  ✓ HTML: ../outputs/act2/act2_fig5_growth_scorecard.html
  ✓ PNG : ../outputs/act2/act2_fig5_growth_scorecard.png

✅  Act II export complete → ../outputs/act2/

Files ready:
   act2_fig1_wage_growth_indexed.html                    17.6 KB
   act2_fig1_wage_growth_indexed.png                    261.0 KB
   act2_fig2_9010_gap.html                               18.4 KB
   act2_fig2_9010_gap.png                               230.9 KB
   act2_fig3_gini.html                                   15.8 KB
   act2_fig3_gini.png                                   216.8 KB
   act2_fig4_tw